In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)


In [ ]:
print("SVM VS LOGISTIC REGRESSION - COMPLAINT CLASSIFIER")
print("\nSTEP 1: Loading dataset...")

data_path = "../data/preprocess_data/complaints_cleaned.csv"
results_dir = "../results/Dev"
os.makedirs(results_dir, exist_ok=True)

df = pd.read_csv(data_path)
print(f"Loaded: {data_path}")
print(f"Shape: {df.shape}")


In [ ]:
print("STEP 2: Preparing features and target...")

target_col = "Product"
primary_feature_col = "Issue"
secondary_feature_candidates = ["Sub-product", "Sub-issue"]
secondary_feature_col = next((c for c in secondary_feature_candidates if c in df.columns), None)

if secondary_feature_col is None:
    raise ValueError(f"Missing secondary feature column. Expected one of: {secondary_feature_candidates}")

required_cols = [primary_feature_col, secondary_feature_col, target_col]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df[primary_feature_col] = df[primary_feature_col].fillna("").astype(str)
df[secondary_feature_col] = df[secondary_feature_col].fillna("").astype(str)
df[target_col] = df[target_col].fillna("").astype(str)

X_text = (df[primary_feature_col] + " " + df[secondary_feature_col]).str.replace(r"\s+", " ", regex=True).str.strip()
y = df[target_col].str.replace(r"\s+", " ", regex=True).str.strip()

valid_mask = (X_text.str.len() > 0) & (y.str.len() > 0)
X_text = X_text[valid_mask]
y = y[valid_mask]

# Cap rows for faster benchmarking while preserving class proportions.
max_rows_for_training = 100000
if len(X_text) > max_rows_for_training:
    X_text, _, y, _ = train_test_split(
        X_text,
        y,
        train_size=max_rows_for_training,
        stratify=y,
        random_state=42,
    )
    print(f"Using stratified sample: {max_rows_for_training:,} rows")

print(f"Features used: {[primary_feature_col, secondary_feature_col]}")
print(f"Target used: {target_col}")
print(f"Rows used: {len(X_text):,}")
print(f"Classes: {y.nunique()}")

# Ensure stratified split is valid (each class needs at least 2 samples).
class_counts = y.value_counts()
low_support_classes = class_counts[class_counts < 2].index.tolist()
if low_support_classes:
    keep_mask = ~y.isin(low_support_classes)
    dropped_rows = int((~keep_mask).sum())
    X_text = X_text[keep_mask]
    y = y[keep_mask]
    print(f"Dropped {dropped_rows} rows from low-support classes: {low_support_classes}")
    print(f"Rows after low-support cleanup: {len(X_text):,}")
    print(f"Classes after cleanup: {y.nunique()}")


In [ ]:
print("STEP 3: Train-test split...")

X_train, X_test, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Train size: {len(X_train):,}")
print(f"Test size: {len(X_test):,}")


In [ ]:
print("STEP 4: Training and evaluating models...")

# Use the same, lighter feature space for both models to keep comparison fair and faster.
vectorizer_kwargs = {
    "ngram_range": (1, 1),
    "min_df": 5,
    "max_df": 0.95,
    "max_features": 50000,
}

models = {
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(**vectorizer_kwargs)),
        ("clf", LogisticRegression(max_iter=700, solver="lbfgs")),
    ]),
    "Linear SVM": Pipeline([
        ("tfidf", TfidfVectorizer(**vectorizer_kwargs)),
        ("clf", LinearSVC(dual="auto", max_iter=3000)),
    ]),
}

metrics_rows = []
reports = {}
predictions = {}
trained_models = {}

for model_name, model in models.items():
    print(f"\nTraining: {model_name}")
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    metrics_rows.append({
        "Model": model_name,
        "Train_Accuracy": accuracy_score(y_train, y_train_pred),
        "Test_Accuracy": accuracy_score(y_test, y_test_pred),
        "Train_Balanced_Accuracy": balanced_accuracy_score(y_train, y_train_pred),
        "Test_Balanced_Accuracy": balanced_accuracy_score(y_test, y_test_pred),
        "Train_Weighted_F1": f1_score(y_train, y_train_pred, average="weighted"),
        "Test_Weighted_F1": f1_score(y_test, y_test_pred, average="weighted"),
        "Test_Macro_F1": f1_score(y_test, y_test_pred, average="macro"),
    })

    reports[model_name] = classification_report(y_test, y_test_pred, zero_division=0)
    predictions[model_name] = y_test_pred
    trained_models[model_name] = model

metrics_df = pd.DataFrame(metrics_rows).set_index("Model")

print("\nEvaluation scores (%):")
print((metrics_df * 100).round(2))


STEP 4: Training and evaluating models...

Training: Logistic Regression

Training: Linear SVM


In [ ]:
print("STEP 5: Plotting model comparisons...")

labels = sorted(y_test.unique())

# 1) Test metric comparison plot
metric_cols = ["Test_Accuracy", "Test_Balanced_Accuracy", "Test_Weighted_F1", "Test_Macro_F1"]
metric_plot_df = (metrics_df[metric_cols] * 100).T
metric_plot_df.plot(kind="bar", figsize=(12, 6))
plt.title("Logistic Regression vs Linear SVM (Test Metrics)")
plt.xlabel("Metric")
plt.ylabel("Score (%)")
plt.ylim(0, 100)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
metric_plot_path = f"{results_dir}/svm_vs_logreg_metric_comparison.png"
plt.savefig(metric_plot_path, dpi=200, bbox_inches="tight")
plt.close()
print(f"Saved: {metric_plot_path}")

# 2) Confusion matrix comparison
fig, axes = plt.subplots(1, 2, figsize=(22, 9))
for ax, (model_name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred, labels=labels)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    cm_norm = np.nan_to_num(cm_norm)

    im = ax.imshow(cm_norm, interpolation="nearest", cmap="Blues", vmin=0, vmax=1)
    ax.set_title(f"{model_name} - Normalized Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_yticklabels(labels, fontsize=7)

fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.02, pad=0.02)
plt.tight_layout()
cm_plot_path = f"{results_dir}/svm_vs_logreg_confusion_matrices.png"
plt.savefig(cm_plot_path, dpi=200, bbox_inches="tight")
plt.close()
print(f"Saved: {cm_plot_path}")

# 3) Actual vs predicted count (top 10 true classes)
top_products = y_test.value_counts().head(10).index
fig, axes = plt.subplots(1, 2, figsize=(22, 7), sharey=True)
for ax, (model_name, y_pred) in zip(axes, predictions.items()):
    actual_counts = y_test.value_counts().reindex(top_products, fill_value=0)
    pred_counts = pd.Series(y_pred).value_counts().reindex(top_products, fill_value=0)
    compare_df = pd.DataFrame({"Actual": actual_counts, "Predicted": pred_counts})
    compare_df.plot(kind="bar", ax=ax)
    ax.set_title(f"{model_name} - Actual vs Predicted (Top 10)")
    ax.set_xlabel("Product")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
actual_pred_plot_path = f"{results_dir}/svm_vs_logreg_actual_vs_predicted.png"
plt.savefig(actual_pred_plot_path, dpi=200, bbox_inches="tight")
plt.close()
print(f"Saved: {actual_pred_plot_path}")


In [ ]:
print("STEP 6: Saving reports and final artifacts...")

scores_path = f"{results_dir}/svm_vs_logreg_scores.csv"
metrics_df.to_csv(scores_path)
print(f"Saved: {scores_path}")

for model_name, report_text in reports.items():
    safe_name = re.sub(r"[^a-z0-9]+", "_", model_name.lower()).strip("_")
    report_path = f"{results_dir}/{safe_name}_classification_report.txt"
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(report_text)
    print(f"Saved: {report_path}")

best_model_name = metrics_df["Test_Weighted_F1"].idxmax()
summary_path = f"{results_dir}/svm_vs_logreg_summary.txt"
with open(summary_path, "w", encoding="utf-8") as f:
    f.write("SVM vs Logistic Regression - Summary\n")
    f.write("===================================\n\n")
    f.write("Scores (fraction):\n")
    f.write(metrics_df.to_string())
    f.write("\n\n")
    f.write(f"Best model by Test Weighted F1: {best_model_name}\n")

print(f"Best model by Test Weighted F1: {best_model_name}")
print(f"Saved: {summary_path}")


In [ ]:
print("STEP 7: Sample prediction with both models...")

sample_issue = "incorrect debit card charge"
sample_secondary_feature = "incorrect charge on my debit card"
sample_feature = f"{sample_issue} {sample_secondary_feature}"

sample_rows = []
for model_name, model in trained_models.items():
    pred_label = model.predict([sample_feature])[0]
    sample_rows.append({
        "Model": model_name,
        "Issue": sample_issue,
        "Secondary Feature": sample_secondary_feature,
        "Predicted Product": pred_label,
    })
    print(f"{model_name} prediction: {pred_label}")

sample_pred_path = f"{results_dir}/svm_vs_logreg_sample_predictions.csv"
pd.DataFrame(sample_rows).to_csv(sample_pred_path, index=False)
print(f"Saved: {sample_pred_path}")

print("\nAll results saved to ../results/Dev")
